In [ ]:
#| default_exp graph

## Notebook symbol graph

Statically match notebook definitions and calls so agents can see which cells define, call, and depend on a symbol.

This notebook adds a static map across notebooks. It does not execute project code; it parses cells with `ast`, records where symbols are defined, and reports which cells call those symbols.

That gives agents a fast way to answer questions like "where is this helper used?" before editing a private function or moving code between notebooks.

The graph is a navigation aid, not a runtime dependency analyzer. It answers practical maintenance questions: where is this symbol defined, who calls it, and are any notebooks importing private helpers that should stay local?

```python
symbol_graph(path="nbs", symbol="write_nb")
private_symbol_report(path="nbs")
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb, write_nb as _write_nb
from nbskill.graph import symbol_graph as _example_symbol_graph
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.mcp import capture_call
from nbskill.graph import private_symbol_report, symbol_graph, symbol_usage_summary

In [ ]:
root = demo_path("10_graph_example")
try:
    root.mkdir()
    _write_nb(_new_nb([
        _mk_cell("#| default_exp demo"),
        _mk_cell("#| export\ndef helper():\n    return 1\n\ndef target():\n    return helper()"),
        _mk_cell("target()"),
    ]), root / "demo.ipynb")
    _example_symbol_graph(str(root), "helper")
finally:
    remove_demo_path(root)

Symbol helper
Definitions:
- nbs/data/10_graph_example/demo.ipynb id=c6025ccd
Callers:
- nbs/data/10_graph_example/demo.ipynb id=c6025ccd
Callees:
- (none)


In [ ]:
#| export
import ast
import builtins
import glob
from pathlib import Path

from fastcore.nbio import read_nb as _read_nb
from fastcore.script import call_parse

from nbskill.foundation import cell_source, cli_error, cli_return, is_export_directive, tracked_call

### Finding notebooks and removing directives

The graph starts with notebook paths. These helpers accept a file, folder, or glob, skip checkpoint folders, and remove nbdev directives before parsing Python so `ast` sees regular source.

In [ ]:
#| export
def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not is_export_directive(line))


def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts


def _notebook_paths(path="nbs"):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file():
        candidates = [pth]
    else:
        candidates = []
    paths = sorted({path for path in candidates if _is_notebook_path(path)})
    if not paths: cli_error(f"No notebooks matched {path!r}")
    return paths


def _graph_scope(path):
    pth = Path(str(path)).expanduser()
    return pth.parent if pth.is_file() else path

### Discovering definitions and calls

The parser records function, class, and method definitions, then walks call expressions inside each parsed tree. Both fully qualified names and short names are kept because notebooks often import helpers into local scope.

In [ ]:
#| export
def _parse_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return None


def _name_parts(node):
    if isinstance(node, ast.Name): return [node.id]
    if isinstance(node, ast.Attribute):
        base = _name_parts(node.value)
        return [*base, node.attr] if base else [node.attr]
    return []


def _call_name(node):
    parts = _name_parts(node)
    return ".".join(parts) if parts else None


def _call_site_records(node, source=""):
    source_lines = str(source or "").splitlines()
    records = []
    for child in ast.walk(node):
        if not isinstance(child, ast.Call): continue
        name = _call_name(child.func)
        if not name: continue
        lineno = getattr(child, "lineno", None)
        line = source_lines[lineno - 1].strip() if lineno and lineno <= len(source_lines) else ""
        for call_name in dict.fromkeys([name, name.rsplit(".", 1)[-1]]):
            records.append({"name": call_name, "lineno": lineno, "line": line})
    return tuple(records)


def _call_names(node):
    return tuple(dict.fromkeys(site["name"] for site in _call_site_records(node)))


def _node_definitions(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return [(node.name, node, "function")]
    if isinstance(node, ast.ClassDef):
        items = [(node.name, node, "class")]
        for child in node.body:
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                items.append((f"{node.name}.{child.name}", child, "method"))
        return items
    return []

### Building graph records

Each code cell can contribute definition records and caller records. The collected graph is deliberately simple dictionaries so reports, tests, and future tools can inspect it without a graph database.

In [ ]:
#| export
def _cell_definition_records(path, module, idx, cell):
    tree = _parse_cell(cell)
    if tree is None: return []
    records = []
    for node in tree.body:
        for symbol, symbol_node, kind in _node_definitions(node):
            records.append({
                "symbol": symbol,
                "kind": kind,
                "module": module,
                "path": str(path),
                "cell_id": getattr(cell, "id", ""),
                "cell_idx": idx,
                "calls": _call_names(symbol_node),
            })
    return records


def _cell_call_record(path, idx, cell):
    source = _source_without_directives(cell_source(cell))
    tree = _parse_cell(cell)
    if tree is None: return None
    calls = _call_names(tree)
    if not calls: return None
    return {
        "path": str(path),
        "cell_id": getattr(cell, "id", ""),
        "cell_idx": idx,
        "calls": calls,
        "call_sites": _call_site_records(tree, source),
    }


def _import_module_name(node):
    if node.module is None: return None
    if node.module == "nbskill": return ""
    if node.module.startswith("nbskill."): return node.module.removeprefix("nbskill.")
    return node.module


def _cell_import_records(path, idx, cell):
    tree = _parse_cell(cell)
    if tree is None: return []
    records = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.ImportFrom): continue
        module = _import_module_name(node)
        if module is None: continue
        for alias in node.names:
            records.append({
                "module": module,
                "symbol": alias.name,
                "local": alias.asname or alias.name,
                "path": str(path),
                "cell_id": getattr(cell, "id", ""),
                "cell_idx": idx,
            })
    return records


def _notebook_module_name(path, nb):
    for cell in nb.cells:
        for line in cell_source(cell).splitlines():
            line = line.strip()
            if line.startswith("#| default_exp "):
                return line.split(None, 2)[-1].replace("/", ".")
    return Path(path).stem


def _collect_graph(path="nbs"):
    definitions, callers, imports = [], [], []
    for nb_path in _notebook_paths(path):
        nb = _read_nb(nb_path)
        module = _notebook_module_name(nb_path, nb)
        for idx, cell in enumerate(nb.cells):
            definitions.extend(_cell_definition_records(nb_path, module, idx, cell))
            imports.extend(_cell_import_records(nb_path, idx, cell))
            record = _cell_call_record(nb_path, idx, cell)
            if record: callers.append(record)
    return {"definitions": definitions, "callers": callers, "imports": imports}

### Cell order warnings

Notebook cells are executed in order, so a top-level call should not appear before the cell that defines or imports its callable. The order checker also looks inside function bodies for callable roots that are never defined or imported, which catches rarely exercised missing imports before runtime.

In [ ]:
#| export
_BUILTIN_CALL_NAMES = set(dir(builtins)) | {"display", "get_ipython"}


def _target_names(target):
    if isinstance(target, ast.Name): return {target.id}
    if isinstance(target, (ast.Tuple, ast.List)):
        names = set()
        for item in target.elts: names.update(_target_names(item))
        return names
    return set()


def _argument_names(args):
    items = [
        *args.posonlyargs, *args.args, *args.kwonlyargs,
        *([args.vararg] if args.vararg else []),
        *([args.kwarg] if args.kwarg else []),
    ]
    return {arg.arg for arg in items if arg is not None}


def _binding_names_from_node(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): return {node.name}
    if isinstance(node, ast.Import): return {alias.asname or alias.name.split(".", 1)[0] for alias in node.names}
    if isinstance(node, ast.ImportFrom):
        if any(alias.name == "*" for alias in node.names): return {"*"}
        return {alias.asname or alias.name for alias in node.names}
    if isinstance(node, ast.Assign):
        names = set()
        for target in node.targets: names.update(_target_names(target))
        return names
    if isinstance(node, ast.AnnAssign): return _target_names(node.target)
    if isinstance(node, (ast.For, ast.AsyncFor)): return _target_names(node.target)
    if isinstance(node, (ast.With, ast.AsyncWith)):
        names = set()
        for item in node.items:
            if item.optional_vars is not None: names.update(_target_names(item.optional_vars))
        return names
    if isinstance(node, ast.ExceptHandler) and node.name: return {node.name}
    return set()


def _body_binding_names(nodes):
    names = set()
    for node in ast.walk(ast.Module(body=list(nodes), type_ignores=[])):
        names.update(_binding_names_from_node(node))
    return names


def _cell_binding_names(tree):
    names = set()
    for node in tree.body: names.update(_binding_names_from_node(node))
    return names


def _call_root_name(node):
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.Attribute): return _call_root_name(node.value)
    return None


class _CallRootVisitor(ast.NodeVisitor):
    def __init__(self):
        self.records = []
        self.local_scopes = []

    def _is_local(self, name):
        return any(name in scope for scope in self.local_scopes)

    def _with_scope(self, names, visit):
        self.local_scopes.append(names)
        visit()
        self.local_scopes.pop()

    def _visit_deferred_body(self, node, body):
        self._with_scope(_argument_names(node.args) | _body_binding_names(body), lambda: [self.visit(child) for child in body])

    def visit_FunctionDef(self, node):
        for item in [*node.decorator_list, *node.args.defaults, *node.args.kw_defaults]:
            if item is not None: self.visit(item)
        if node.returns is not None: self.visit(node.returns)
        self._visit_deferred_body(node, node.body)

    def visit_AsyncFunctionDef(self, node): self.visit_FunctionDef(node)

    def visit_Lambda(self, node):
        self._with_scope(_argument_names(node.args), lambda: self.visit(node.body))

    def _visit_comprehension(self, node):
        names = set()
        for generator in node.generators: names.update(_target_names(generator.target))
        def visit_body():
            for generator in node.generators:
                self.visit(generator.iter)
                for item in generator.ifs: self.visit(item)
            if hasattr(node, "elt"): self.visit(node.elt)
            if hasattr(node, "key"): self.visit(node.key)
            if hasattr(node, "value"): self.visit(node.value)
        self._with_scope(names, visit_body)

    def visit_ListComp(self, node): self._visit_comprehension(node)
    def visit_SetComp(self, node): self._visit_comprehension(node)
    def visit_DictComp(self, node): self._visit_comprehension(node)
    def visit_GeneratorExp(self, node): self._visit_comprehension(node)

    def visit_Call(self, node):
        name = _call_root_name(node.func)
        if name and not self._is_local(name):
            self.records.append({"symbol": name, "line": getattr(node, "lineno", None), "deferred": bool(self.local_scopes)})
        self.generic_visit(node)


def _cell_call_root_records(cell):
    tree = _parse_cell(cell)
    if tree is None: return []
    visitor = _CallRootVisitor()
    visitor.visit(tree)
    return visitor.records


def _cell_order_data(path):
    data = []
    for nb_path in _notebook_paths(path):
        try:
            nb = _read_nb(nb_path)
        except FileNotFoundError:
            continue
        cells = []
        for idx, cell in enumerate(nb.cells):
            tree = _parse_cell(cell)
            bindings = set() if tree is None else _cell_binding_names(tree)
            calls = _cell_call_root_records(cell)
            cells.append({"idx": idx, "cell": cell, "bindings": bindings, "calls": calls})
        data.append({"path": nb_path, "cells": cells})
    return data


def _binding_locations(cells):
    locations = {}
    for item in cells:
        for name in item["bindings"]:
            locations.setdefault(name, []).append(item)
    return locations


def _first_later_binding(locations, name, idx):
    return next((item for item in locations.get(name, []) if item["idx"] > idx), None)


def _order_problem(kind, nb_path, item, call, detail, confidence="medium"):
    return {
        "code": kind,
        "path": str(nb_path),
        "cell_id": getattr(item["cell"], "id", ""),
        "line": call.get("line"),
        "symbol": call["symbol"],
        "detail": detail,
        "severity": "warning",
        "source": "nbskill",
        "confidence": confidence,
    }


def _format_order_problem(problem):
    line = f" line={problem['line']}" if problem.get("line") else ""
    return f"- {problem['code']}: {problem['path']} id={problem.get('cell_id', '')}{line} symbol={problem['symbol']!r} {problem['detail']}"


def notebook_order_problems(path="nbs"):
    "Return structured calls-before-definitions and missing callable import warnings."
    problems = []
    for nb_data in _cell_order_data(path):
        nb_path, cells = nb_data["path"], nb_data["cells"]
        locations = _binding_locations(cells)
        available = set(_BUILTIN_CALL_NAMES)
        for item in cells:
            cell_available = available | item["bindings"]
            star_imported = "*" in cell_available
            for call in item["calls"]:
                symbol = call["symbol"]
                if symbol in cell_available or star_imported: continue
                later = _first_later_binding(locations, symbol, item["idx"])
                if later and not call.get("deferred"):
                    problems.append(_order_problem(
                        "cell-order", nb_path, item, call,
                        f"called before definition/import in later cell id={getattr(later['cell'], 'id', '')}",
                        confidence="high",
                    ))
                elif not later:
                    problems.append(_order_problem(
                        "missing-import", nb_path, item, call,
                        "called without an earlier definition/import",
                        confidence="medium" if call.get("deferred") else "high",
                    ))
            available.update(item["bindings"])
    seen, unique = set(), []
    for problem in problems:
        key = tuple(problem.get(item) for item in ("code", "path", "cell_id", "line", "symbol"))
        if key in seen: continue
        seen.add(key)
        unique.append(problem)
    return unique


def notebook_order_problem_lines(path="nbs"):
    "Return style-check lines for calls before definitions/imports and missing callable imports."
    return [_format_order_problem(problem) for problem in notebook_order_problems(path)]

### Resolving symbol relationships

The graph is approximate by design. Matching by exact or short symbol name is enough to surface likely callers and callees, which is the useful pre-edit signal for this project.

In [ ]:
#| export
def _symbol_short_name(symbol):
    return str(symbol).rsplit(".", 1)[-1]


def _call_matches_symbol(call, symbol):
    call = str(call)
    symbol = str(symbol)
    return call == symbol or _symbol_short_name(call) == _symbol_short_name(symbol)


def _definitions_for_symbol(graph, symbol):
    return [record for record in graph["definitions"] if record["symbol"] == symbol or _symbol_short_name(record["symbol"]) == symbol]


def _caller_records_for_symbol(graph, symbol):
    return [record for record in graph["callers"] if any(_call_matches_symbol(call, symbol) for call in record["calls"])]


def _resolve_callees(graph, calls):
    symbols = {record["symbol"] for record in graph["definitions"]}
    resolved = []
    for call in calls:
        for symbol in symbols:
            if _call_matches_symbol(call, symbol): resolved.append(symbol)
    return sorted(set(resolved))


def _locations(records):
    return [f"{record['path']} id={record['cell_id']}" for record in records]


def _callee_locations(graph, symbol):
    return _locations(_definitions_for_symbol(graph, symbol))

### Formatting reports

The reporting helpers turn graph records into compact text. They are meant for agent context: locations include notebook paths and cell ids so the next step can jump straight to `nb_cell` or `show_doc`.

In [ ]:
#| export
def _symbol_graph_data(path, symbol):
    graph = _collect_graph(_graph_scope(path))
    definitions = _definitions_for_symbol(graph, symbol)
    callers = _caller_records_for_symbol(graph, symbol)
    callee_symbols = []
    for definition in definitions:
        callee_symbols.extend(_resolve_callees(graph, definition["calls"]))
    callee_symbols = sorted(set(item for item in callee_symbols if item != symbol))
    return {"symbol": symbol, "definitions": definitions, "callers": callers, "callees": callee_symbols, "graph": graph}


def _caller_usage_lines(records, symbol, limit=8):
    lines, seen = [], set()
    for record in records:
        for site in record.get("call_sites", ()):
            if not _call_matches_symbol(site.get("name"), symbol): continue
            key = (record["path"], record["cell_id"], site.get("lineno"), site.get("line"))
            if key in seen: continue
            seen.add(key)
            lineno = f" line {site['lineno']}" if site.get("lineno") else ""
            source = f": {site['line']}" if site.get("line") else ""
            lines.append(f"- {record['path']} id={record['cell_id']}{lineno}{source}")
            if len(lines) >= limit: return lines
    return lines


def _format_symbol_graph_data(data):
    lines = [f"Symbol {data['symbol']}"]
    lines.append("Definitions:")
    lines.extend([f"- {loc}" for loc in _locations(data["definitions"]) ] or ["- (none)"])
    lines.append("Callers:")
    lines.extend([f"- {loc}" for loc in _locations(data["callers"]) ] or ["- (none)"])
    caller_usage = _caller_usage_lines(data["callers"], data["symbol"])
    if caller_usage:
        lines.append("Caller usages:")
        lines.extend(caller_usage)
    lines.append("Callees:")
    if data["callees"]:
        for symbol in data["callees"]:
            locs = "; ".join(_callee_locations(data["graph"], symbol)) or "unknown location"
            lines.append(f"- {symbol}: {locs}")
    else:
        lines.append("- (none)")
    return "\n".join(lines)


def symbol_usage_summary(path, symbols):
    "Return a compact caller/callee summary for one or more symbols."
    if isinstance(symbols, str): symbols = [symbols]
    chunks = []
    for symbol in dict.fromkeys(symbols or []):
        data = _symbol_graph_data(path, symbol)
        callers = "; ".join(_locations(data["callers"])[:8]) or "none"
        callees = "; ".join(data["callees"][:8]) or "none"
        chunks.append(f"{symbol}: callers={callers}; callees={callees}")
        caller_usage = _caller_usage_lines(data["callers"], symbol)
        if caller_usage:
            chunks.append("Caller usages:")
            chunks.extend(caller_usage)
    return "\n".join(chunks)

### Public graph reports

`symbol_graph` focuses on one symbol's definitions, callers, and callees. `private_symbol_report` looks for cross-notebook calls to private helpers, which is useful when deciding whether a private function can be changed safely.

In [ ]:
#| export
@call_parse
@tracked_call
def symbol_graph(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
    symbol: str = "",  # Function, class, or Class.method to inspect
):
    "Print definitions, callers, and callees for a notebook symbol."
    if not symbol: cli_error("Pass --symbol to inspect")
    text = _format_symbol_graph_data(_symbol_graph_data(path, symbol))
    print(text)
    return cli_return(text)


@call_parse
@tracked_call
def private_symbol_report(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
):
    "Print cross-notebook calls to imported private `_` symbols."
    graph = _collect_graph(path)
    definitions = {(record.get("module"), record["symbol"]): record for record in graph["definitions"]}
    lines = ["Cross-notebook private symbol calls"]
    for imported in graph["imports"]:
        symbol = imported["symbol"]
        if not _symbol_short_name(symbol).startswith("_"): continue
        definition = definitions.get((imported["module"], symbol))
        if not definition or definition["path"] == imported["path"]: continue
        for caller in graph["callers"]:
            if caller["path"] != imported["path"]: continue
            if not any(_call_matches_symbol(call, imported["local"]) for call in caller["calls"]): continue
            lines.append(
                f"- {symbol} defined {definition['path']} id={definition['cell_id']} "
                f"called from {caller['path']} id={caller['cell_id']}"
            )
    if len(lines) == 1: lines.append("No cross-notebook private symbol calls found.")
    text = "\n".join(dict.fromkeys(lines))
    print(text)
    return cli_return(text)

In [ ]:
root = demo_path("10_graph_tests")
try:
    root.mkdir()
    lib = root / "lib.ipynb"
    caller = root / "caller.ipynb"
    _write_nb(_new_nb([
        _mk_cell("#| default_exp lib"),
        _mk_cell("#| export\ndef helper():\n    return 1\n\ndef _secret():\n    return helper()\n\ndef target():\n    return helper()"),
    ]), lib)
    _write_nb(_new_nb([
        _mk_cell("from nbskill.lib import target, _secret\nvalue = target()\n_secret()"),
    ]), caller)
    text = capture_call(symbol_graph, path=str(root), symbol="target")
    assert "Definitions:" in text
    assert "caller.ipynb id=" in text
    assert "Caller usages:" in text
    assert "value = target()" in text
    assert "helper" in text
    summary = symbol_usage_summary(str(root), ["target"])
    assert "callers=" in summary and "caller.ipynb id=" in summary
    assert "Caller usages:" in summary and "value = target()" in summary
    order_nb = root / "order.ipynb"
    _write_nb(_new_nb([
        _mk_cell("result = later_helper()"),
        _mk_cell("def later_helper():\n    return 1"),
        _mk_cell("def loader():\n    return MissingPath('x')"),
    ]), order_nb)
    order_lines = notebook_order_problem_lines(str(order_nb))
    assert any("cell-order" in line and "later_helper" in line for line in order_lines)
    assert any("missing-import" in line and "MissingPath" in line for line in order_lines)
    report = capture_call(private_symbol_report, path=str(root))
    assert "_secret" in report
    assert "called from" in report
finally:
    remove_demo_path(root)